# 🔎 RAG-поиск по научным статьям: от OpenAlex до оценки RAGAS

**Лабораторная работа «RAG-поиск» · Вариант 2 (исследовательский уровень)**

Полный конвейер *Retrieval-Augmented Generation* на корпусе реальных научных публикаций по теме
**«Retrieval-Augmented Generation & Vector Databases»**: сбор корпуса через REST API → две стратегии чанкинга →
эмбеддинги и FAISS → устойчивый к опечаткам препроцессинг запросов → LLM с заземлением и цитатами →
количественная оценка поиска (Recall/Precision/HitRate/MRR) и генерации (RAGAS, Abstention, Hallucination Rate).

```
 OpenAlex API ──► корпус (≥300 статей, .jsonl) ──► "Title + Abstract"
                                                     │
                       ┌─────────────────────────────┴──────────────────────────┐
                       ▼                                                        ▼
        Стратегия A: скользящее окно (Python)              Стратегия B: RecursiveCharacterTextSplitter
                       └─────────────────────────────┬──────────────────────────┘
                                                     ▼  bge-small / MiniLM, L2-нормализация
                                              FAISS IndexFlatIP
 запрос ─► исправление опечаток (белый список) ─► top-k ─► порог уверенности ─► LLM + строгий промпт ─► ответ с [цитатами]
```

**Как запустить.** Colab → `Среда выполнения → Сменить среду → T4 GPU` → `Выполнить всё`.
Полный прогон занимает ~20-30 минут на T4. Без GPU тоже работает, но генерация LLM будет медленной.
RAGAS (Шаг 7.2) включается автоматически, если в секретах Colab (🔑 слева) задан `JUDGE_API_KEY`
(подходит бесплатный ключ [Groq](https://console.groq.com/keys)). Без ключа считаются все остальные метрики.

**Оглавление**
0. Установка и конфигурация
1. Сбор корпуса (OpenAlex)
2. Препроцессинг и две стратегии чанкинга
3. Эмбеддинги и индекс FAISS
4. Обработка запросов: исправление опечаток без порчи терминов
5. LLM, системный промпт, заземление и 5 тестовых сценариев
6. Количественная оценка поиска (IR Evaluation)
7. Оценка генерации: RAGAS, Abstention, Hallucination Rate, ablation по top-k
8. Автоматический отчёт и выводы

## 0. Установка и конфигурация

Все параметры эксперимента собраны в одной ячейке, чтобы работу было легко воспроизвести или перенести на другую тему:
достаточно поменять `SEARCH_QUERIES` и `TOPIC_KEYWORDS`.

Версии LangChain-пакетов закреплены на ветке `0.3`, потому что именно с ней совместим `ragas==0.2.x`.

In [ ]:
!pip install -q faiss-cpu rapidfuzz sentence-transformers tabulate "langchain-text-splitters>=0.3,<0.4"

In [ ]:
import os, re, json, time, random, math, textwrap, warnings, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---------------- Конфигурация эксперимента ----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)

TOPIC_NAME = "Retrieval-Augmented Generation & Vector Databases"
SEARCH_QUERIES = [
    "retrieval-augmented generation",
    "retrieval augmented large language models",
    "vector database",
    "dense passage retrieval",
    "approximate nearest neighbor search embeddings",
    "hybrid retrieval BM25 dense",
    "hallucination grounding retrieval LLM",
    "text embedding retrieval benchmark",
]
# слова-маркеры темы: статья без них считается шумом поисковой выдачи
TOPIC_KEYWORDS = ["retriev", "rag", "vector", "embedding", "nearest neighbor",
                  "search", "knowledge-intensive", "question answering", "index"]
PER_QUERY_LIMIT = 150        # сколько статей брать максимум с одного запроса
MAX_DOCS = 600               # верхняя граница размера корпуса
MIN_DOCS = 300               # требование задания
MIN_ABSTRACT_WORDS = 60
YEAR_FROM = 2019
OPENALEX_MAILTO = ""         # можно указать e-mail -> "polite pool" OpenAlex (быстрее и стабильнее)
ARXIV_SOURCE_ID = "S4306400194"   # arXiv (Cornell University) в OpenAlex

EMB_MODEL_MAIN = "BAAI/bge-small-en-v1.5"
EMB_MODELS_COMPARE = ["BAAI/bge-small-en-v1.5", "sentence-transformers/all-MiniLM-L6-v2"]
# у BGE для запросов рекомендована инструкция-префикс (документы кодируются без неё)
QUERY_PREFIX = {"BAAI/bge-small-en-v1.5": "Represent this sentence for searching relevant passages: "}

LLM_BACKEND = "local"        # "local" (transformers) или "api" (любой OpenAI-совместимый эндпоинт)
LOCAL_LLM = "Qwen/Qwen2.5-1.5B-Instruct"
API_BASE_URL = "https://api.groq.com/openai/v1"
API_MODEL = "llama-3.1-8b-instant"
MAX_NEW_TOKENS = 400

# LLM-судья для RAGAS (OpenAI-совместимый API; ключ берётся из секретов Colab / переменных окружения)
JUDGE_BASE_URL = "https://api.groq.com/openai/v1"
JUDGE_MODEL = "llama-3.3-70b-versatile"

TOP_K = 5
REFUSAL = "Информация в найденных статьях отсутствует"

DATA_DIR, RES_DIR, ART_DIR = Path("data"), Path("results"), Path("artifacts")
FIG_DIR = RES_DIR / "figures"
for d in (DATA_DIR, RES_DIR, FIG_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)

def get_secret(name):
    """Секрет из Colab userdata, иначе из переменной окружения."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
print("Устройство:", DEVICE)

def savefig(name):
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Сохранено:", path)

## 1. Сбор корпуса документов (OpenAlex API)

**Почему OpenAlex.** Это открытый каталог научных работ с REST API без капчи и жёстких лимитов по IP; arXiv-препринты
в нём индексируются как отдельный источник, поэтому можно фильтровать по `primary_location.source.id` arXiv.

**Что делает код:**
* `requests.Session` + `Retry` с экспоненциальной задержкой автоматически повторяет запрос при `429/5xx` и обрывах
  соединения; каждый поисковый запрос обёрнут в `try/except`, так что сбой одного запроса не роняет весь сбор;
* пагинация через **cursor** (`cursor=*` → `meta.next_cursor`), а не через номер страницы: так OpenAlex рекомендует
  обходить большие выдачи;
* `select=` запрашивает только нужные поля, это в разы уменьшает объём ответа;
* аннотация хранится в OpenAlex как **инвертированный индекс** `{слово: [позиции]}`, его нужно собрать обратно в текст;
* если по запросу с фильтром arXiv нашлось мало статей, делается повторный запрос без фильтра источника;
* дедупликация идёт по `doc_id` и по нормализованному заголовку (одна статья может встречаться под разными записями);
* тематический фильтр по `TOPIC_KEYWORDS` убирает шум полнотекстового поиска;
* результат кэшируется в `data/corpus.jsonl`: повторный запуск не ходит в сеть и даёт ровно тот же корпус.

In [ ]:
def make_session():
    s = requests.Session()
    retry = Retry(total=6, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET"], respect_retry_after_header=True)
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update({"User-Agent": f"rag-lab/1.0 (mailto:{OPENALEX_MAILTO or 'anonymous'})"})
    return s

def reconstruct_abstract(inverted_index: dict) -> str:
    """Восстановление текста аннотации из abstract_inverted_index."""
    if not inverted_index:
        return ""
    positions = [(pos, word) for word, poss in inverted_index.items() for pos in poss]
    positions.sort(key=lambda x: x[0])
    return " ".join(w for _, w in positions)

ARXIV_RE = re.compile(r"(?:arxiv\.org/(?:abs|pdf)/|arxiv\.)(\d{4}\.\d{4,5})", re.I)

def extract_arxiv_id(work: dict):
    loc = work.get("primary_location") or {}
    candidates = [loc.get("landing_page_url"), loc.get("pdf_url"), work.get("doi"),
                  (work.get("ids") or {}).get("doi")]
    for c in candidates:
        if c:
            m = ARXIV_RE.search(c)
            if m:
                return m.group(1)
    return None

def clean_text(s: str) -> str:
    s = re.sub(r"<[^>]+>", " ", s or "")                  # html-теги
    s = re.sub(r"^\s*(abstract|summary)[\s.:—-]*", "", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip()

def parse_work(work: dict):
    title = clean_text(work.get("title") or work.get("display_name") or "")
    abstract = clean_text(reconstruct_abstract(work.get("abstract_inverted_index")))
    arxiv_id = extract_arxiv_id(work)
    openalex_id = (work.get("id") or "").rsplit("/", 1)[-1]
    if arxiv_id:
        doc_id, url = f"arxiv_{arxiv_id}", f"https://arxiv.org/abs/{arxiv_id}"
    else:
        loc = work.get("primary_location") or {}
        doc_id = f"openalex_{openalex_id}"
        url = loc.get("landing_page_url") or work.get("doi") or work.get("id")
    authors = [a.get("author", {}).get("display_name") for a in (work.get("authorships") or [])]
    return {
        "doc_id": doc_id,
        "title": title,
        "authors": [a for a in authors if a],
        "year": int(work.get("publication_year") or 0),
        "abstract": abstract,
        "url": url,
        "cited_by_count": int(work.get("cited_by_count") or 0),
    }

def fetch_openalex(session, query, limit, arxiv_only=True):
    filters = [f"publication_year:>{YEAR_FROM - 1}", "has_abstract:true"]
    if arxiv_only:
        filters.append(f"primary_location.source.id:{ARXIV_SOURCE_ID}")
    params = {
        "search": query,
        "filter": ",".join(filters),
        "per_page": 100,
        "cursor": "*",
        "select": "id,doi,title,display_name,publication_year,authorships,"
                  "abstract_inverted_index,primary_location,ids,cited_by_count",
    }
    if OPENALEX_MAILTO:
        params["mailto"] = OPENALEX_MAILTO
    works = []
    while len(works) < limit:
        r = session.get("https://api.openalex.org/works", params=params, timeout=30)
        r.raise_for_status()
        payload = r.json()
        batch = payload.get("results", [])
        if not batch:
            break
        works.extend(batch)
        params["cursor"] = (payload.get("meta") or {}).get("next_cursor")
        if not params["cursor"]:
            break
        time.sleep(0.3)   # бережём API: ~3 запроса в секунду
    return works[:limit]

def norm_title(t):
    return re.sub(r"[^a-z0-9]", "", t.lower())

def is_on_topic(rec):
    text = (rec["title"] + " " + rec["abstract"]).lower()
    return any(k in text for k in TOPIC_KEYWORDS)

CORPUS_PATH = DATA_DIR / "corpus.jsonl"

if CORPUS_PATH.exists():
    corpus_df = pd.read_json(CORPUS_PATH, lines=True, dtype={"doc_id": str})
    print(f"Корпус загружен из кэша: {len(corpus_df)} статей")
else:
    session = make_session()
    records, seen_ids, seen_titles, log = [], set(), set(), []
    # раунд 2 запускается автоматически, только если после раунда 1 статей меньше MIN_DOCS
    for round_no, limit in enumerate([PER_QUERY_LIMIT, PER_QUERY_LIMIT * 3], start=1):
        if round_no > 1 and len(records) >= MIN_DOCS:
            break
        print(f"--- Раунд {round_no}: до {limit} работ на запрос ---")
        for q in SEARCH_QUERIES:
            raw = []
            try:
                raw = fetch_openalex(session, q, limit, arxiv_only=True)
                mode = "arXiv"
                if len(raw) < 30:   # мало препринтов — расширяем поиск на все источники
                    raw += fetch_openalex(session, q, limit, arxiv_only=False)
                    mode = "arXiv+all"
            except Exception as e:
                mode = f"ошибка: {type(e).__name__}"
            added = 0
            for w in raw:
                rec = parse_work(w)
                if (len(rec["abstract"].split()) < MIN_ABSTRACT_WORDS or not rec["title"]
                        or not is_on_topic(rec)):
                    continue
                key = norm_title(rec["title"])
                if rec["doc_id"] in seen_ids or key in seen_titles:
                    continue
                seen_ids.add(rec["doc_id"]); seen_titles.add(key)
                records.append(rec); added += 1
            log.append({"раунд": round_no, "query": q, "режим": mode, "получено": len(raw), "добавлено новых": added})
            print(f"{q!r:55} {mode:10} получено={len(raw):4}  новых={added:4}  всего={len(records)}")
            if len(records) >= MAX_DOCS:
                break
    corpus_df = pd.DataFrame(records[:MAX_DOCS])
    pd.DataFrame(log).to_csv(RES_DIR / "collection_log.csv", index=False)
    if len(corpus_df) >= MIN_DOCS:   # кэшируем только полноценный корпус
        corpus_df.to_json(CORPUS_PATH, orient="records", lines=True, force_ascii=False)
        print(f"\nСохранено {len(corpus_df)} уникальных статей -> {CORPUS_PATH}")

assert len(corpus_df) >= MIN_DOCS, (
    f"Собрано только {len(corpus_df)} статей < {MIN_DOCS}: добавьте запросы в SEARCH_QUERIES "
    f"или увеличьте PER_QUERY_LIMIT, затем удалите data/corpus.jsonl")
assert corpus_df["doc_id"].is_unique
corpus_df.head(5)[["doc_id", "title", "year", "authors", "url"]]

### 1.1. Разведочный анализ корпуса

Длина аннотаций определяет параметры чанкинга, поэтому смотрим на неё до выбора размера окна.

In [ ]:
corpus_df["abstract_words"] = corpus_df["abstract"].str.split().str.len()
corpus_stats = {
    "Статей": len(corpus_df),
    "Из arXiv (doc_id arxiv_*)": int(corpus_df["doc_id"].str.startswith("arxiv_").sum()),
    "Годы": f"{corpus_df['year'].min()}-{corpus_df['year'].max()}",
    "Слов в аннотации (медиана)": int(corpus_df["abstract_words"].median()),
    "Слов в аннотации (90-й перцентиль)": int(corpus_df["abstract_words"].quantile(0.9)),
    "Слов в аннотации (макс.)": int(corpus_df["abstract_words"].max()),
}
display(pd.Series(corpus_stats, name="значение").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
corpus_df["year"].value_counts().sort_index().plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Статьи по годам"); axes[0].set_xlabel("")
axes[1].hist(corpus_df["abstract_words"], bins=30, color="#55A868")
axes[1].axvline(130, color="crimson", ls="--", label="окно стратегии A = 130 слов")
axes[1].set_title("Длина аннотации, слов"); axes[1].legend()
plt.tight_layout(); savefig("01_corpus_eda"); plt.show()

## 2. Препроцессинг и две стратегии чанкинга

Базовый текст статьи: `"Title: " + title + "\n\nAbstract: " + abstract`.

Аннотации короткие (обычно 150-250 слов), поэтому статья разбивается на 1-3 фрагмента. Без названия «хвостовой»
фрагмент аннотации теряет контекст: фраза *«we propose a two-stage method…»* ничего не говорит о том, о какой статье
идёт речь. Поэтому к каждому чанку, который не начинается с названия, добавляется заголовок `Title: …`.

| | Стратегия A: скользящее окно | Стратегия B: Recursive Character Splitter |
|---|---|---|
| Единица | слова | символы |
| Размер / перекрытие | 130 / 25 слов (шаг S = 105) | 700 / 100 символов |
| Границы | фиксированные, могут резать предложения | ищет `\n\n` → `\n` → `. ` → пробел, старается сохранить предложения |
| Реализация | чистый Python | LangChain |

**Почему такие размеры.** 130 слов ≈ 170-200 токенов WordPiece: это помещается в лимит 256 токенов all-MiniLM-L6-v2
(у bge-small лимит 512), то есть модель видит чанк целиком и ничего не обрезает. 700 символов ≈ 110-120 слов, поэтому
обе стратегии сопоставимы по объёму. Перекрытие ~20% страхует от разрыва мысли на границе окна.

**Деталь реализации A.** Последнее окно выравнивается по концу текста (`words[-chunk_size:]`). Иначе при длине
140 слов получился бы «огрызок» из 35 слов, 25 из которых уже есть в предыдущем чанке.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def document_text(row):
    return f"Title: {row['title']}\n\nAbstract: {row['abstract']}"

def chunk_text_custom(text, chunk_size_words=130, overlap_words=25):
    """Стратегия A: скользящее окно по словам с шагом S = chunk_size - overlap."""
    assert 0 <= overlap_words < chunk_size_words
    words = text.split()
    if len(words) <= chunk_size_words:
        return [" ".join(words)]
    step = chunk_size_words - overlap_words
    chunks, start = [], 0
    while True:
        end = start + chunk_size_words
        if end >= len(words):                      # последнее окно выравниваем по концу
            chunks.append(" ".join(words[-chunk_size_words:]))
            break
        chunks.append(" ".join(words[start:end]))
        start += step
    return chunks

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700, chunk_overlap=100, separators=["\n\n", "\n", ". ", " ", ""])

def chunk_text_recursive(text):
    return recursive_splitter.split_text(text)

def make_chunks(df, strategy, add_title=True):
    """Возвращает список чанков с привязкой к родительскому документу и метаданным."""
    chunks = []
    for _, row in df.iterrows():
        text = document_text(row)
        pieces = chunk_text_custom(text) if strategy == "A" else chunk_text_recursive(text)
        for i, piece in enumerate(pieces):
            piece = piece.strip()
            if add_title and not piece.startswith("Title:"):
                piece = f"Title: {row['title']}\n\n{piece}"
            if not add_title and i > 0:
                piece = piece.replace(f"Title: {row['title']}", "").strip()
            chunks.append({
                "chunk_id": f"{row['doc_id']}_c{i:02d}",
                "doc_id": row["doc_id"],
                "text": piece,
                "metadata": {"title": row["title"], "authors": row["authors"], "year": int(row["year"]),
                             "url": row["url"], "strategy": strategy, "position": i},
            })
    return chunks

chunk_sets = {
    "A": make_chunks(corpus_df, "A"),
    "B": make_chunks(corpus_df, "B"),
}
print(json.dumps(chunk_sets["A"][1], ensure_ascii=False, indent=2)[:900])

In [ ]:
def chunk_stats(chunks, name):
    lens = np.array([len(c["text"].split()) for c in chunks])
    per_doc = pd.Series([c["doc_id"] for c in chunks]).value_counts()
    return {"стратегия": name, "чанков": len(chunks), "чанков на статью (сред.)": round(per_doc.mean(), 2),
            "статей с 1 чанком, %": round(100 * (per_doc == 1).mean(), 1),
            "слов в чанке (медиана)": int(np.median(lens)), "мин": int(lens.min()), "макс": int(lens.max())}

chunk_stats_df = pd.DataFrame([chunk_stats(chunk_sets["A"], "A: окно 130/25 слов"),
                               chunk_stats(chunk_sets["B"], "B: recursive 700/100 симв.")])
display(chunk_stats_df)

fig, ax = plt.subplots(figsize=(8, 3.4))
for name, color in [("A", "#4C72B0"), ("B", "#DD8452")]:
    ax.hist([len(c["text"].split()) for c in chunk_sets[name]], bins=40, alpha=0.6, label=name, color=color)
ax.set_title("Распределение длины чанков (слов, с учётом префикса Title)"); ax.legend()
plt.tight_layout(); savefig("02_chunk_lengths"); plt.show()

## 3. Векторизация и индекс FAISS

* **Би-энкодер** кодирует запрос и документ независимо, поэтому все чанки можно закодировать заранее, а при поиске
  считать только один вектор запроса. Основная модель `BAAI/bge-small-en-v1.5` (d = 384), для сравнения
  `all-MiniLM-L6-v2` (d = 384).
* **L2-нормализация.** `IndexFlatIP` ищет по скалярному произведению. Для единичных векторов
  $\mathbf{u}\cdot\mathbf{v} = \cos(\mathbf{u},\mathbf{v})$, поэтому после `faiss.normalize_L2` скор лежит в $[-1, 1]$
  и означает косинусную близость. Без нормализации наверх всплывали бы векторы с большой нормой, а не самые похожие.
  Ниже это проверяется явно: печатаются нормы до и после.
* **`IndexFlatIP`** выполняет точный перебор. Для ~1-2 тыс. векторов это миллисекунды; HNSW/IVF нужны от сотен тысяч.
* Для каждой стратегии (и каждой модели) строится **изолированный** индекс.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

_embedders = {}
def get_embedder(name):
    if name not in _embedders:
        _embedders[name] = SentenceTransformer(name, device=DEVICE)
    return _embedders[name]

def encode(model_name, texts, is_query=False, batch_size=64):
    prefix = QUERY_PREFIX.get(model_name, "") if is_query else ""
    emb = get_embedder(model_name).encode([prefix + t for t in texts], batch_size=batch_size,
                                          convert_to_numpy=True, show_progress_bar=False)
    return np.ascontiguousarray(emb, dtype="float32")

def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    embeddings = embeddings.copy()
    faiss.normalize_L2(embeddings)              # L2-нормализация на месте
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

def retrieve(query: str, model_name, index, chunks_list, top_k: int = 5):
    """Top-k чанков с метаданными и косинусным скором в [-1, 1]."""
    q = encode(model_name, [query], is_query=True)
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, top_k)
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), start=1):
        if idx != -1:
            c = chunks_list[idx]
            results.append({"rank": rank, "score": float(score), "chunk_id": c["chunk_id"],
                            "doc_id": c["doc_id"], "title": c["metadata"]["title"],
                            "year": c["metadata"]["year"], "url": c["metadata"]["url"],
                            "text": c["text"], "chunk": c})
    return results

class VectorStore:
    """Связка: модель эмбеддингов + чанки + индекс FAISS."""
    def __init__(self, model_name, chunks, name=""):
        self.model_name, self.chunks, self.name = model_name, chunks, name
        t0 = time.time()
        self.raw_emb = encode(model_name, [c["text"] for c in chunks])
        self.index = build_faiss_index(self.raw_emb)
        self.build_seconds = time.time() - t0

    def retrieve(self, query, top_k=TOP_K):
        return retrieve(query, self.model_name, self.index, self.chunks, top_k)

    def search_batch(self, queries, k):
        q = encode(self.model_name, queries, is_query=True)
        faiss.normalize_L2(q)
        return self.index.search(q, k)

stores = {s: VectorStore(EMB_MODEL_MAIN, chunk_sets[s], name=f"{s} · {EMB_MODEL_MAIN}") for s in ("A", "B")}
for s, st in stores.items():
    faiss.write_index(st.index, str(ART_DIR / f"index_{s}.faiss"))
    norms_before = np.linalg.norm(st.raw_emb, axis=1)
    norms_after = np.linalg.norm(st.index.reconstruct_n(0, st.index.ntotal), axis=1)
    print(f"Стратегия {s}: {st.index.ntotal} векторов, d={st.index.d}, построено за {st.build_seconds:.1f} с | "
          f"норма до: {norms_before.min():.3f}…{norms_before.max():.3f}, после: {norms_after.min():.4f}…{norms_after.max():.4f}")

In [ ]:
def show_results(results, width=110):
    return pd.DataFrame([{"rank": r["rank"], "score": round(r["score"], 4), "chunk_id": r["chunk_id"],
                          "title": r["title"][:70], "year": r["year"],
                          "фрагмент": textwrap.shorten(r["text"].split("\n\n", 1)[-1], width)} for r in results])

demo_query = "How can retrieval reduce hallucinations of large language models?"
for s in ("A", "B"):
    print(f"\nСтратегия {s} — запрос: {demo_query!r}")
    display(show_results(stores[s].retrieve(demo_query, top_k=5)))

## 4. Обработка пользовательских запросов: исправление опечаток

**Проблема.** Обычный спеллчекер знает бытовой словарь и «чинит» научные термины: `LoRA → Lord`, `RAPTOR → raptor`,
`ColBERT → colbert`. А опечатки вида `retreival`, `generaton` нужно исправлять, иначе эмбеддинг запроса «уплывает».

**Решение:**
1. **Доменный словарь** строится из всех токенов длиной > 2 по названиям и аннотациям корпуса (с частотами).
   К нему добавляется общий английский лексикон NLTK `words` и проверка словоформ (`questions` → `question`):
   обычное слово, которого случайно нет в аннотациях, не является опечаткой.
2. **Белый список**: токен не трогаем, если он (или его основа) есть в словаре, в списке аббревиатур, написан КАПСОМ, содержит
   заглавную букву внутри слова (`LoRA`, `ColBERT`), цифру (`GPT-4`, `top-k`) или короче 4 символов.
3. **Исправление**: кандидат ищется среди слов словаря с частотой ≥ 2 по расстоянию **Дамерау-Левенштейна**
   (`rapidfuzz`; перестановка соседних букв = 1 операция). Порог: ≤ 1 для слов до 5 букв, ≤ 2 для длинных
   (иначе короткие слова превращаются в другие короткие слова). При равном расстоянии выбирается более частое слово.
4. Регистр и пунктуация исходного токена сохраняются.

In [ ]:
from rapidfuzz.distance import DamerauLevenshtein
from rapidfuzz import fuzz

TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9\-]*")
DOMAIN_ABBREVIATIONS = {
    "RAG", "LLM", "LLMs", "NLP", "QA", "ODQA", "KILT", "BEIR", "MTEB", "MS-MARCO", "BM25", "TF-IDF", "DPR", "ColBERT",
    "ColBERTv2", "ANCE", "SPLADE", "FAISS", "HNSW", "IVF", "PQ", "ANN", "kNN", "LoRA", "QLoRA", "RLHF", "GPT", "BERT",
    "RoBERTa", "T5", "BART", "REALM", "RETRO", "Atlas", "FiD", "Self-RAG", "CRAG", "RAPTOR", "GraphRAG", "HyDE", "RAGAS",
    "Milvus", "Qdrant", "Weaviate", "Pinecone", "Chroma", "pgvector", "ScaNN", "DiskANN", "GNN", "CoT", "MoE",
}
ABBR_LOWER = {a.lower() for a in DOMAIN_ABBREVIATIONS}
# служебные слова вопросов могут не встречаться в аннотациях, но исправлять их не нужно
QUESTION_WORDS = {"what", "which", "who", "whom", "why", "how", "when", "where", "does", "did", "do", "is", "are",
                  "was", "were", "can", "could", "should", "would", "compare", "explain", "describe", "list",
                  "summarize", "main", "between", "versus", "difference", "differences", "papers", "collected"}

token_freq = Counter()
for text in (corpus_df["title"] + " " + corpus_df["abstract"]):
    token_freq.update(t.lower() for t in TOKEN_RE.findall(text) if len(t) > 2)

# Общий английский лексикон: обычные слова вопроса («reaction», «encode») могут не встречаться в аннотациях,
# но это не опечатки. Без него корректор превратил бы «reaction» в доменное «reduction».
try:
    import nltk
    nltk.download("words", quiet=True)
    from nltk.corpus import words as nltk_words
    GENERAL_LEXICON = {w.lower() for w in nltk_words.words()}
except Exception:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    GENERAL_LEXICON = set(ENGLISH_STOP_WORDS)
print(f"Общий английский лексикон: {len(GENERAL_LEXICON)} слов")

DOMAIN_VOCAB = set(token_freq) | ABBR_LOWER | QUESTION_WORDS | GENERAL_LEXICON   # белый список
SUFFIXES = ("ies", "es", "s", "ed", "ing", "ly", "er", "ers")

def in_vocab(tok, vocab):
    """Слово или его основа (без типичного суффикса) есть в словаре: questions → question, reduces → reduce."""
    w = tok.lower()
    if w in vocab:
        return True
    for suf in SUFFIXES:
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            stem = w[: -len(suf)]
            if stem in vocab or stem + "e" in vocab or (suf == "ies" and stem + "y" in vocab):
                return True
    return False
CORRECTION_TARGETS = {w: f for w, f in token_freq.items() if f >= 2 and w.isalpha()}
TARGETS_BY_LEN = defaultdict(list)
for w, f in CORRECTION_TARGETS.items():
    TARGETS_BY_LEN[len(w)].append((w, f))
print(f"Доменный словарь: {len(DOMAIN_VOCAB)} токенов, кандидатов для исправления: {len(CORRECTION_TARGETS)}")
print("Топ-25 слов корпуса:", ", ".join(w for w, _ in token_freq.most_common(25)))

def is_protected(tok, vocab=DOMAIN_VOCAB):
    return (in_vocab(tok, vocab) or tok in DOMAIN_ABBREVIATIONS or tok.lower() in ABBR_LOWER
            or tok.isupper() or any(ch.isupper() for ch in tok[1:]) or any(ch.isdigit() for ch in tok)
            or len(tok) <= 3)

def best_correction(word):
    w = word.lower()
    max_d = 1 if len(w) <= 5 else 2
    best = None   # (distance, -freq, candidate)
    for L in range(len(w) - max_d, len(w) + max_d + 1):
        for cand, freq in TARGETS_BY_LEN.get(L, ()):
            d = DamerauLevenshtein.distance(w, cand, score_cutoff=max_d)
            if d <= max_d and (best is None or (d, -freq) < best[:2]):
                best = (d, -freq, cand)
    return (best[2], best[0]) if best else (None, None)

def match_case(src, dst):
    if src.isupper():
        return dst.upper()
    if src[:1].isupper():
        return dst[:1].upper() + dst[1:]
    return dst

def clean_and_correct_query(query: str, protect=True, return_changes=False):
    query = re.sub(r"\s+", " ", query).strip()
    out, changes = [], []
    for token in query.split(" "):
        m = re.match(r"^(\W*)(.*?)(\W*)$", token)
        pre, core, post = m.groups()
        sub_tokens = core.split("-") if "-" in core and not is_protected(core) else [core]
        fixed_parts = []
        for part in sub_tokens:
            if not part or not part.isascii() or not part.isalpha() or (protect and is_protected(part)):
                fixed_parts.append(part); continue
            if not protect and part.lower() in CORRECTION_TARGETS:
                fixed_parts.append(part); continue
            cand, dist = best_correction(part)
            if cand and cand != part.lower():
                fixed = match_case(part, cand)
                changes.append({"исходное": part, "исправлено": fixed, "DL-расстояние": dist})
                fixed_parts.append(fixed)
            else:
                fixed_parts.append(part)
        out.append(pre + "-".join(fixed_parts) + post)
    result = " ".join(out)
    return (result, changes) if return_changes else result

In [ ]:
typo_demo = [
    "How does retreival-augmented generaton reduce halucinations?",
    "dense pasage retrival with dual encoders",
    "vectr databse indexing for aproximate neighbor serch",
    "LoRA fine-tuning of the retriever and generator",      # термины должны остаться нетронутыми
    "ColBERT late interaction vs BM25 in RAPTOR and Self-RAG",
    "HNSW graph index in FAISS for kNN search",
]
rows = []
for q in typo_demo:
    fixed, changes = clean_and_correct_query(q, return_changes=True)
    naive = clean_and_correct_query(q, protect=False)
    rows.append({"запрос": q, "после очистки (с белым списком)": fixed,
                 "исправления": "; ".join(f"{c['исходное']}→{c['исправлено']} (d={c['DL-расстояние']})" for c in changes) or "—",
                 "наивно (без защиты терминов)": naive})
typo_demo_df = pd.DataFrame(rows)
display(typo_demo_df)

protected_terms = ["LoRA", "RAG", "ColBERT", "HNSW", "FAISS", "RAPTOR", "Self-RAG", "BM25", "kNN", "GraphRAG"]
protected_check = pd.DataFrame([{"термин": t, "после очистки": clean_and_correct_query(t),
                                 "сохранён": clean_and_correct_query(t) == t,
                                 "наивный режим": clean_and_correct_query(t, protect=False)} for t in protected_terms])
display(protected_check)
assert protected_check["сохранён"].all(), "Белый список должен сохранять доменные термины"

**Что видно в таблицах.** С белым списком опечатки в обычных словах исправляются в доменные слова корпуса,
а аббревиатуры и смешанный регистр остаются нетронутыми (проверено `assert`). В наивном режиме (без белого списка)
корректор начинает «лечить» всё, чего нет среди частых слов, и портит часть терминов. Это и есть типичная ошибка №4
из методички.

## 5. Интеграция LLM и промпт-инжиниринг

**Генератор.** `Qwen2.5-1.5B-Instruct` локально (fp16 на T4 занимает ~3,5 ГБ, квантование не требуется) или любая
модель через OpenAI-совместимый API (`LLM_BACKEND = "api"`). Генерация детерминированная (`do_sample=False`),
поэтому результаты воспроизводимы.

**Три уровня защиты от галлюцинаций:**
1. **Строгий системный промпт** (шаблон из методички): только факты из контекста, цитаты `[Название (Год)]`, при
   отсутствии информации ровно одна фраза-отказ.
2. **Порог уверенности поиска τ.** Если косинусный скор лучшего чанка ниже τ, вопрос почти наверняка вне корпуса,
   и пайплайн отказывает сразу, не вызывая LLM. Маленькие модели (1-2 млрд параметров) склонны отвечать «из памяти»,
   даже когда промпт это запрещает, поэтому детерминированный фильтр надёжнее. τ калибруется на **отдельном** наборе
   вопросов, не пересекающемся с тестовыми сценариями.
3. **Пост-проверка ответа**: нормализация отказа к точной фразе и проверка, что каждая цитата ссылается на статью,
   которая действительно была в контексте (*citation validity*).

In [ ]:
SYSTEM_PROMPT = f"""Вы — строгий академический ассистент по анализу научных публикаций.
Ваша задача — ответить на вопрос пользователя, опираясь ИСКЛЮЧИТЕЛЬНО на предоставленный контекст из научных статей.

ПРАВИЛА ОТВЕТА:
1. Используйте ТОЛЬКО факты, явно упомянутые в контексте. Не делайте предположений и не привлекайте внешние знания.
2. Для каждого утверждения в ответе укажите источник в формате: [Название статьи (Год)]. Название копируйте из контекста без перевода.
3. Если в предоставленном контексте нет прямой информации для ответа на вопрос, вы ОБЯЗАНЫ вывести ровно одну фразу:
«{REFUSAL}».
4. Отвечайте на русском языке, кратко и по существу (не более 8 предложений)."""

USER_TEMPLATE = """КОНТЕКСТ:
{context}

ВОПРОС:
{question}

ОТВЕТ:"""

PLAIN_SYSTEM = "Вы — полезный ассистент. Отвечайте на русском языке, кратко и по существу (не более 8 предложений)."

class LLM:
    def __init__(self, backend=LLM_BACKEND):
        self.backend = backend
        if backend == "local":
            from transformers import pipeline
            import torch
            self.pipe = pipeline("text-generation", model=LOCAL_LLM, device_map="auto",
                                 torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
            self.name = LOCAL_LLM
        else:
            from openai import OpenAI
            self.client = OpenAI(base_url=API_BASE_URL, api_key=get_secret("LLM_API_KEY"))
            self.name = API_MODEL

    def chat(self, system, user, max_new_tokens=MAX_NEW_TOKENS):
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        if self.backend == "local":
            out = self.pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
            return out[0]["generated_text"][-1]["content"].strip()
        for attempt in range(5):
            try:
                r = self.client.chat.completions.create(model=self.name, messages=messages,
                                                        temperature=0, max_tokens=max_new_tokens)
                return r.choices[0].message.content.strip()
            except Exception:
                time.sleep(2 ** attempt)
        return ""

llm = LLM()
print("Генератор:", llm.name)

In [ ]:
CITATION_RE = re.compile(r"\[([^\[\]]+?)\s*\((\d{4})\)\]")

def format_context(results):
    blocks = []
    for i, r in enumerate(results, 1):
        authors = ", ".join(r["chunk"]["metadata"]["authors"][:3]) or "n/a"
        body = r["text"].split("\n\n", 1)[-1] if r["text"].startswith("Title:") else r["text"]
        blocks.append(f"[Источник {i}] {r['title']} ({r['year']})\nАвторы: {authors}\nТекст: {body}")
    return "\n\n".join(blocks)

def check_citations(answer, results):
    cited = CITATION_RE.findall(answer)
    ctx = [(r["title"], r["year"]) for r in results]
    valid = [any(fuzz.token_set_ratio(t.lower(), ct.lower()) >= 85 and int(y) == cy for ct, cy in ctx)
             for t, y in cited]
    return cited, valid

def normalize_refusal(answer):
    a = answer.strip().strip("«»\"' ")
    if a.lower().startswith(REFUSAL.lower()) or (REFUSAL.lower() in a.lower() and not CITATION_RE.search(a)):
        return REFUSAL, True
    return answer.strip(), False

class RAGPipeline:
    def __init__(self, store, llm, tau=None):
        self.store, self.llm, self.tau = store, llm, tau

    def answer(self, question, top_k=TOP_K, correct=True, use_gate=True):
        q_clean, changes = clean_and_correct_query(question, return_changes=True) if correct else (question, [])
        results = self.store.retrieve(q_clean, top_k=top_k)
        top1 = results[0]["score"] if results else -1.0
        gated = use_gate and self.tau is not None and top1 < self.tau
        if gated:
            raw = REFUSAL
        else:
            raw = self.llm.chat(SYSTEM_PROMPT, USER_TEMPLATE.format(context=format_context(results), question=q_clean))
        answer, refused = normalize_refusal(raw)
        cited, valid = check_citations(answer, results)
        return {"question": question, "query_used": q_clean, "corrections": changes, "results": results,
                "contexts": [r["text"] for r in results], "top1_score": top1, "gated": gated,
                "answer": answer, "refused": refused, "citations": cited, "citations_valid": valid}

def plain_llm_answer(question):
    return llm.chat(PLAIN_SYSTEM, question)

### 5.1. Калибровка порога уверенности τ

Два небольших набора вопросов: явно по теме корпуса и явно вне его. Порог выбирается так, чтобы максимально
разделить их по скору top-1 (при ничьей берётся меньший τ: лучше пропустить вопрос к LLM, у которой есть второй
барьер в виде промпта, чем ошибочно отказать). Тестовые сценарии в калибровке **не участвуют**.

In [ ]:
CALIB_IN = [
    "What is retrieval-augmented generation?",
    "How are dense retrievers trained with contrastive learning?",
    "Which benchmarks are used to evaluate text embedding models?",
    "How do vector databases index high-dimensional embeddings?",
    "How can retrieved passages be fused in the reader model?",
    "What are the limitations of BM25 compared to neural retrieval?",
    "How does query rewriting improve retrieval for LLMs?",
    "How is retrieval used in open-domain question answering?",
    "What methods compress or quantize embedding vectors for search?",
    "How can knowledge graphs be combined with retrieval-augmented generation?",
]
CALIB_OUT = [
    "What is the boiling point of ethanol at sea level?",
    "Who won the football World Cup in 2014?",
    "How do I bake sourdough bread at home?",
    "What are the symptoms of iron deficiency anemia?",
    "Explain the plot of Shakespeare's Hamlet.",
    "What is the capital of Australia?",
    "How does photosynthesis work in C4 plants?",
    "What is the best exercise routine for building muscle?",
    "How are volcanoes formed by plate tectonics?",
    "What is the tax rate on capital gains in Germany?",
]

def top1_scores(store, questions):
    s, _ = store.search_batch([clean_and_correct_query(q) for q in questions], 1)
    return s[:, 0]

def calibrate_tau(store):
    s_in, s_out = top1_scores(store, CALIB_IN), top1_scores(store, CALIB_OUT)
    allv = np.sort(np.concatenate([s_in, s_out]))
    cands = np.concatenate([[allv[0] - 1e-3], (allv[1:] + allv[:-1]) / 2, [allv[-1] + 1e-3]])
    acc = [((s_in >= t).sum() + (s_out < t).sum()) / (len(s_in) + len(s_out)) for t in cands]
    best = max(acc)
    tau = float(min(t for t, a in zip(cands, acc) if a == best))
    return tau, best, s_in, s_out

MAIN_STRATEGY = "A"   # уточняется по результатам Шага 6; τ откалиброван для основного индекса
tau, calib_acc, s_in, s_out = calibrate_tau(stores[MAIN_STRATEGY])
print(f"τ = {tau:.4f}, точность разделения на калибровке = {calib_acc:.0%}")

plt.figure(figsize=(8, 3))
plt.hist(s_in, bins=12, alpha=0.7, label="по теме корпуса", color="#55A868")
plt.hist(s_out, bins=12, alpha=0.7, label="вне корпуса", color="#C44E52")
plt.axvline(tau, color="black", ls="--", label=f"τ = {tau:.3f}")
plt.title("Скор top-1 для калибровочных вопросов"); plt.xlabel("cosine similarity"); plt.legend()
plt.tight_layout(); savefig("05_tau_calibration"); plt.show()

rag = RAGPipeline(stores[MAIN_STRATEGY], llm, tau=tau)

### 5.2. Пять обязательных сценариев тестирования

Вопросы задаются на английском, потому что корпус и модель эмбеддингов англоязычные, а ответ генерируется на русском.
Для каждого сценария показываем найденные источники, ответ RAG и ответ «чистой» LLM без контекста.

In [ ]:
SCENARIOS = [
    ("1. Фактический", "How does dense passage retrieval encode questions and passages, and how is it trained?"),
    ("2. Сравнительный", "Compare sparse lexical retrieval such as BM25 with dense embedding-based retrieval: "
                         "what are the strengths and weaknesses of each?"),
    ("3. С опечатками", "How does retreival-augmented generaton help reduce halucinations of large langage models?"),
    ("4. Out-of-domain", "What is the mechanism of the SN2 nucleophilic substitution reaction in organic chemistry?"),
    ("5. Обобщение", "What are the main approaches to evaluating retrieval-augmented generation systems "
                     "discussed in the collected papers?"),
]

scenario_runs = []
for name, q in SCENARIOS:
    run = rag.answer(q)
    run["scenario"] = name
    run["plain_answer"] = plain_llm_answer(q)
    scenario_runs.append(run)

    print("=" * 110)
    print(f"СЦЕНАРИЙ {name}\nВопрос: {q}")
    if run["corrections"]:
        print("Исправлено:", "; ".join(f"{c['исходное']}→{c['исправлено']}" for c in run["corrections"]))
        print("Запрос в поиск:", run["query_used"])
    print(f"top-1 score = {run['top1_score']:.3f} (τ = {tau:.3f}){'  → ОТКАЗ ПО ПОРОГУ' if run['gated'] else ''}")
    display(show_results(run["results"], width=80)[["rank", "score", "title", "year"]])
    print("\n🟢 RAG:\n" + textwrap.fill(run["answer"], 110))
    print(f"   цитат: {len(run['citations'])}, валидных: {sum(run['citations_valid'])}")
    print("\n⚪ LLM без контекста:\n" + textwrap.fill(run["plain_answer"], 110))

### 5.3. Сравнительная таблица «чистая LLM vs RAG»

Колонка «Автопроверка» считается программно: число и валидность цитат, корректность отказа, наличие ссылок у чистой
LLM. Она даёт объективную основу для экспертного комментария под таблицей.

In [ ]:
def auto_comment(run):
    n, v = len(run["citations"]), sum(run["citations_valid"])
    ood = run["scenario"].startswith("4")
    if ood:
        rag_part = ("RAG корректно отказал" + (" (по порогу τ)" if run["gated"] else " (решение LLM)")
                    if run["refused"] else "RAG НЕ отказал: ответ не заземлён — галлюцинация")
        plain_part = ("чистая LLM ответила из собственной памяти без источников"
                      if not normalize_refusal(run["plain_answer"])[1] else "чистая LLM тоже отказалась")
        return f"{rag_part}; {plain_part}."
    if run["refused"]:
        return "RAG отказал на вопросе по теме (ложный отказ) — проверить retrieval/порог."
    cite = f"цитат {n}, валидных {v}" + (" — все ссылки ведут на статьи из контекста" if n and v == n else
                                         (" — есть ссылки вне контекста" if n else " — ответ без цитат"))
    return f"RAG: {cite}; чистая LLM: ссылок на источники нет, проверить факты невозможно."

comparison_df = pd.DataFrame([{
    "Сценарий": r["scenario"],
    "Вопрос": r["question"],
    "Ответ чистой LLM": r["plain_answer"],
    "Ответ RAG (LLM + FAISS)": r["answer"],
    "Источники (top-3)": "; ".join(f"{x['title'][:50]} ({x['year']})" for x in r["results"][:3]),
    "Автопроверка": auto_comment(r),
} for r in scenario_runs])
comparison_df.to_csv(RES_DIR / "scenarios_comparison.csv", index=False)
display(comparison_df.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"}))

**Экспертный комментарий (как читать результаты).**

* **Фактический и сравнительный вопросы (1-2).** Чистая LLM отвечает гладко, но «в среднем по интернету»: без
  источников, иногда с выдуманными деталями (названия статей, числа, авторы). RAG отвечает уже, зато каждое
  утверждение привязано к `[Название (Год)]`, и программная проверка показывает, ведут ли цитаты на статьи из контекста.
  Сравнительный вопрос у RAG собирается из *разных* статей: это видно по разным заголовкам в top-3.
* **Опечатки (3).** До очистки запрос с `retreival`, `generaton`, `halucinations` даёт смещённый эмбеддинг;
  модуль из Шага 4 восстанавливает термины, и поиск возвращает тематически точные статьи. Количественно эффект
  измерен в Шаге 6.3.2.
* **Out-of-domain (4).** Главная проверка на галлюцинации. Чистая LLM уверенно рассказывает про SN2-реакцию: ответ
  может быть даже верным, но он не заземлён в корпусе, а значит недопустим для системы, которая обязана отвечать
  только по документам. RAG выдаёт точную фразу-отказ. Порог τ делает этот отказ детерминированным и не зависит от
  того, послушается ли маленькая LLM промпта.
* **Обобщение (5).** Самый трудный тип для RAG: top-5 чанков дают лишь срез корпуса, поэтому ответ перечисляет
  подходы из найденных статей, а не «все тренды». Это честное ограничение: для полноценного обзора нужен больший
  `top_k` или map-reduce суммаризация по кластерам статей.

## 6. Количественная оценка качества поиска (IR Evaluation)

### 6.1. Офлайн-датасет

* $N = 150$ случайных статей (seed = 42); запрос $q$ — **только название статьи**; релевантные $\mathrm{Rel}_q$ —
  все чанки этой статьи.
* **Почему задача нетривиальна.** Название конкурирует с ~1000+ чанками остальных статей, многие из которых про то же
  самое (RAG, retrieval, embeddings). Тематически близкие статьи с похожими формулировками вытесняют истинные фрагменты
  из топа. Кроме того, «хвостовые» чанки содержат название лишь как префикс, а основную массу текста составляет середина
  аннотации, семантически далёкая от короткого заголовка.
* **Контроль утечки (Query Leakage).** Эмбеддинг запроса считается только от названия, а документа — от связки
  «Title + Abstract». Чтобы честно измерить, какая часть качества объясняется совпадением названия, добавлен
  **абляционный индекс `A-noTitle`**: та же стратегия A, но без префикса `Title:` в хвостовых чанках.

### 6.2. Метрики (для k ∈ {1, 3, 5, 10})

$\mathrm{Recall@k} = \frac{|R_k \cap Rel|}{|Rel|}$, $\mathrm{Precision@k} = \frac{|R_k \cap Rel|}{k}$,
$\mathrm{HitRate@k} = \mathbb{I}(|R_k \cap Rel| > 0)$, $\mathrm{MRR@k} = 1/\mathrm{rank}_{\text{первого релевантного}}$
(0, если в top-k его нет), а также средний скор top-1 и разрыв $\Delta_k = s_1 - s_k$.

У статьи обычно 1-3 чанка, поэтому Precision@k при больших k ограничена сверху: при |Rel| = 2 максимум
Precision@10 = 0,2. Для наглядности дополнительно считается нормированная $\mathrm{nPrecision@k} = \frac{|R_k \cap Rel|}{\min(k, |Rel|)}$.

In [ ]:
KS = (1, 3, 5, 10)
K_MAX = max(KS)
N_EVAL = 150

eval_docs = corpus_df.sample(n=min(N_EVAL, len(corpus_df)), random_state=SEED).reset_index(drop=True)

def evaluate_retrieval(store, queries, true_doc_ids, ks=KS):
    scores, idxs = store.search_batch(queries, max(ks))
    chunk_doc = np.array([c["doc_id"] for c in store.chunks])
    rel_count = Counter(chunk_doc)
    rows = []
    for qi, (doc_id, row_idx, row_sc) in enumerate(zip(true_doc_ids, idxs, scores)):
        is_rel = chunk_doc[row_idx] == doc_id
        n_rel = rel_count[doc_id]
        first = int(np.argmax(is_rel)) + 1 if is_rel.any() else None
        rec = {"doc_id": doc_id, "n_rel": n_rel, "first_rel_rank": first, "top1_score": float(row_sc[0])}
        for k in ks:
            hits = int(is_rel[:k].sum())
            rec[f"Recall@{k}"] = hits / n_rel
            rec[f"Precision@{k}"] = hits / k
            rec[f"nPrecision@{k}"] = hits / min(k, n_rel)
            rec[f"HitRate@{k}"] = float(hits > 0)
            rec[f"MRR@{k}"] = 1.0 / first if first and first <= k else 0.0
            if k > 1:
                rec[f"Margin@{k}"] = float(row_sc[0] - row_sc[k - 1])
        rows.append(rec)
    return pd.DataFrame(rows)

def summarize(per_q, label):
    metrics = ["Recall", "Precision", "nPrecision", "HitRate", "MRR"]
    table = pd.DataFrame({f"@{k}": [per_q[f"{m}@{k}"].mean() for m in metrics] for k in KS}, index=metrics)
    table.loc["Margin s1−sk"] = [np.nan] + [per_q[f"Margin@{k}"].mean() for k in KS[1:]]
    table.loc["Mean top-1 score"] = [per_q["top1_score"].mean()] + [np.nan] * (len(KS) - 1)
    table.columns.name = label
    return table

### 6.3.1. Сравнение стратегий чанкинга (и моделей эмбеддингов)

Сетка экспериментов: {A, B, A-noTitle} × {bge-small, MiniLM}. Индексы для основной модели уже построены,
остальные собираем здесь.

In [ ]:
chunk_sets["A-noTitle"] = make_chunks(corpus_df, "A", add_title=False)
grid_rows, per_query_all, summaries = [], {}, {}
queries = eval_docs["title"].tolist()
true_ids = eval_docs["doc_id"].tolist()

for model_name in EMB_MODELS_COMPARE:
    short = model_name.split("/")[-1]
    for strat in ("A", "B", "A-noTitle"):
        st = stores[strat] if (model_name == EMB_MODEL_MAIN and strat in stores) else VectorStore(model_name, chunk_sets[strat])
        if model_name == EMB_MODEL_MAIN:
            stores.setdefault(strat, st)
        pq = evaluate_retrieval(st, queries, true_ids)
        per_query_all[(short, strat)] = pq
        summaries[(short, strat)] = summarize(pq, f"{strat} · {short}")
        grid_rows.append({"модель": short, "стратегия": strat, "чанков": len(st.chunks),
                          **{f"Recall@{k}": pq[f"Recall@{k}"].mean() for k in (1, 5, 10)},
                          **{f"HitRate@{k}": pq[f"HitRate@{k}"].mean() for k in (1, 5)},
                          "MRR@10": pq["MRR@10"].mean(), "Mean top-1": pq["top1_score"].mean()})

grid_df = pd.DataFrame(grid_rows)
grid_df.to_csv(RES_DIR / "ir_grid.csv", index=False)
display(grid_df.style.format(precision=3).background_gradient(subset=["Recall@5", "MRR@10"], cmap="Greens"))

main_short = EMB_MODEL_MAIN.split("/")[-1]
for strat in ("A", "B"):
    t = summaries[(main_short, strat)]
    t.to_csv(RES_DIR / f"ir_metrics_{strat}.csv")
    print(f"\nСтратегия {strat} · {main_short}: метрика × k")
    display(t.style.format(precision=3, na_rep="—"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, metric in zip(axes, ["Recall", "HitRate", "MRR"]):
    for strat, color in [("A", "#4C72B0"), ("B", "#DD8452"), ("A-noTitle", "#8C8C8C")]:
        pq = per_query_all[(main_short, strat)]
        ax.plot(KS, [pq[f"{metric}@{k}"].mean() for k in KS], marker="o", label=strat, color=color)
    ax.set_title(f"{metric}@k · {main_short}"); ax.set_xticks(KS); ax.set_xlabel("k"); ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout(); savefig("06_ir_strategies"); plt.show()

best_row = grid_df[grid_df["модель"] == main_short].set_index("стратегия").loc[["A", "B"]]
BEST_STRATEGY = best_row["MRR@10"].idxmax()
print(f"Лучшая стратегия по MRR@10 для {main_short}: {BEST_STRATEGY}")

### 6.3.2. Влияние пайплайна очистки запросов

40 названий из тестового набора программно искажаются: в 1-2 словах длиной ≥ 5 букв выполняется одна из операций
(замена буквы на соседнюю по клавиатуре, удаление гласной, перестановка соседних букв). Сравниваем `Recall@5` и
`MRR@10` на чистых, искажённых и очищенных запросах. Дополнительно считаем, какая доля искажённых слов восстановлена
**точно**.

> ⚠️ Честная оговорка: словарь корректора построен по всему корпусу, включая слова из этих же названий. Это
> соответствует реальному сценарию (пользователь ищет по известной коллекции), но немного завышает точность
> восстановления по сравнению с запросами про незнакомые термины.

In [ ]:
KEYBOARD_NEIGHBORS = {c: n for row in ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
                      for i, c in enumerate(row) for n in [row[max(0, i - 1):i] + row[i + 1:i + 2]]}
VOWELS = set("aeiou")

def corrupt_word(word, rng):
    ops = ["sub", "swap"] + (["del_vowel"] if any(ch in VOWELS for ch in word[1:-1]) else [])
    op = rng.choice(ops)
    w = list(word)
    if op == "del_vowel":
        pos = rng.choice([i for i in range(1, len(w) - 1) if w[i].lower() in VOWELS])
        del w[pos]
    elif op == "swap":
        i = rng.randrange(1, len(w) - 2) if len(w) > 3 else 0
        w[i], w[i + 1] = w[i + 1], w[i]
    else:
        i = rng.randrange(1, len(w))
        ch = w[i].lower()
        w[i] = rng.choice(KEYBOARD_NEIGHBORS.get(ch, "e") or "e")
    return "".join(w)

def corrupt_title(title, rng):
    words = title.split()
    cand = [i for i, t in enumerate(words) if t.isalpha() and len(t) >= 5 and not t.isupper()]
    if not cand:
        return title, []
    chosen = rng.sample(cand, k=min(len(cand), rng.choice([1, 2])))
    truth = []
    for i in chosen:
        bad = corrupt_word(words[i], rng)
        truth.append((words[i], bad)); words[i] = bad
    return " ".join(words), truth

rng = random.Random(SEED)
typo_set = eval_docs.head(40).copy()
typo_set[["noisy", "truth"]] = typo_set["title"].apply(lambda t: pd.Series(corrupt_title(t, rng)))
typo_set["corrected"] = typo_set["noisy"].apply(clean_and_correct_query)

restored = total = 0
for _, r in typo_set.iterrows():
    corr_tokens = set(re.findall(r"\w+", r["corrected"].lower()))
    for orig, _bad in r["truth"]:
        total += 1
        restored += orig.lower().strip(".,:;") in corr_tokens
word_restore_rate = restored / max(total, 1)

typo_store = stores[BEST_STRATEGY]
typo_results = []
for label, col in [("чистые названия", "title"), ("с опечатками (до очистки)", "noisy"), ("после очистки", "corrected")]:
    pq = evaluate_retrieval(typo_store, typo_set[col].tolist(), typo_set["doc_id"].tolist())
    typo_results.append({"запросы": label, "Recall@5": pq["Recall@5"].mean(), "MRR@10": pq["MRR@10"].mean(),
                         "HitRate@1": pq["HitRate@1"].mean()})
typo_df = pd.DataFrame(typo_results)
typo_df.to_csv(RES_DIR / "typo_robustness.csv", index=False)
print(f"Индекс: стратегия {BEST_STRATEGY} · {main_short}. Точно восстановлено слов: "
      f"{restored}/{total} = {word_restore_rate:.0%}")
display(typo_df.style.format(precision=3))
display(typo_set[["title", "noisy", "corrected"]].head(8))

fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(2); w = 0.27
for i, (row, color) in enumerate(zip(typo_df.itertuples(), ["#55A868", "#C44E52", "#4C72B0"])):
    ax.bar(x + (i - 1) * w, [row[2], row[3]], w, label=row[1], color=color)
ax.set_xticks(x, ["Recall@5", "MRR@10"]); ax.set_ylim(0, 1.05); ax.legend(loc="lower right")
ax.set_title("Устойчивость поиска к опечаткам")
plt.tight_layout(); savefig("06_typo_robustness"); plt.show()

**Если лучшая стратегия по Шагу 6 отличается от основной**, пересобираем пайплайн на ней и заново калибруем τ,
чтобы Шаг 7 оценивал лучшую конфигурацию.

In [ ]:
if BEST_STRATEGY != MAIN_STRATEGY:
    MAIN_STRATEGY = BEST_STRATEGY
    tau, calib_acc, s_in, s_out = calibrate_tau(stores[MAIN_STRATEGY])
    rag = RAGPipeline(stores[MAIN_STRATEGY], llm, tau=tau)
print(f"Пайплайн для Шага 7: стратегия {MAIN_STRATEGY}, τ = {tau:.4f}")

## 7. Оценка качества генерации

### 7.1. Тестовый бенчмарк (20 сценариев)

* **16 вопросов с ответом в корпусе**: для случайных статей LLM по аннотации генерирует пару
  «вопрос (EN) — эталонный ответ (RU)». Мы знаем статью-источник каждого вопроса, поэтому можем дополнительно
  проверить, нашёл ли её поиск.
* **4 вопроса без ответа** (посторонние темы, не пересекаются с калибровкой τ и сценарием 4); эталон —
  фраза-отказ.

In [ ]:
QA_GEN_PROMPT = """Below is the abstract of a scientific paper. Write ONE specific question in English that can be
answered from this abstract alone (do not mention "this paper" or "the authors"; mention the method or problem explicitly),
and a reference answer in Russian (1-3 sentences) based strictly on the abstract.
Return ONLY JSON: {{"question": "...", "answer": "..."}}

TITLE: {title}
ABSTRACT: {abstract}"""

def parse_json_obj(text):
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if obj.get("question") and obj.get("answer") else None
    except Exception:
        return None

EVAL_SET_PATH = RES_DIR / "eval_set.json"
if EVAL_SET_PATH.exists():
    eval_set = json.loads(EVAL_SET_PATH.read_text(encoding="utf-8"))
else:
    eval_set = []
    pool = corpus_df.sample(frac=1.0, random_state=SEED + 1)
    for attempt, (_, row) in enumerate(pool.iterrows()):
        if len(eval_set) >= 16 or attempt >= 80:   # страховка от бесконечных попыток
            break
        obj = parse_json_obj(llm.chat("You generate evaluation data. Output valid JSON only.",
                                      QA_GEN_PROMPT.format(title=row["title"], abstract=row["abstract"]),
                                      max_new_tokens=300))
        if obj:
            eval_set.append({"question": obj["question"].strip(), "ground_truth": obj["answer"].strip(),
                             "source_doc_id": row["doc_id"], "source_title": row["title"], "in_corpus": True})
    for q in ["What were the main causes of the French Revolution?",
              "How do you treat a sprained ankle at home?",
              "What is the chemical composition of Portland cement?",
              "Which grape varieties are used in Champagne production?"]:
        eval_set.append({"question": q, "ground_truth": REFUSAL, "source_doc_id": None,
                         "source_title": None, "in_corpus": False})
    EVAL_SET_PATH.write_text(json.dumps(eval_set, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Вопросов: {len(eval_set)} (в корпусе: {sum(e['in_corpus'] for e in eval_set)}, "
      f"вне корпуса: {sum(not e['in_corpus'] for e in eval_set)})")
pd.DataFrame(eval_set)[["question", "ground_truth", "in_corpus"]].head(6)

### 7.2. Прогон пайплайна для top_k ∈ {1, 3, 5} и метрики без судьи

Эти метрики считаются всегда, без внешнего API:
* **Abstention Accuracy**: доля вопросов вне корпуса, на которые выдан корректный отказ;
* **Hallucination Rate**: доля вопросов вне корпуса, на которые система выдала содержательный (выдуманный) ответ, $= 1 - \text{Abstention}$;
* **False Refusal Rate**: доля вопросов *с ответом в корпусе*, на которые система ошибочно отказала (обратная сторона строгости);
* **Source Hit@k**: попала ли статья-источник вопроса в контекст (прокси Context Recall, не требующий LLM-судьи);
* **Citation Validity**: доля цитат, ведущих на статьи из поданного контекста;
* **Source Cited**: доля ответов, где процитирована именно статья-источник.

In [ ]:
ABLATION_KS = (1, 3, 5)
gen_runs = {}
for k in ABLATION_KS:
    rows = []
    for e in eval_set:
        run = rag.answer(e["question"], top_k=k)
        rows.append({**e, "k": k, "answer": run["answer"], "contexts": run["contexts"], "refused": run["refused"],
                     "gated": run["gated"], "top1_score": run["top1_score"],
                     "retrieved_doc_ids": [r["doc_id"] for r in run["results"]],
                     "n_citations": len(run["citations"]), "n_valid_citations": sum(run["citations_valid"]),
                     "cited_titles": [t for t, _ in run["citations"]]})
    gen_runs[k] = pd.DataFrame(rows)

def own_metrics(df):
    ind, ood = df[df["in_corpus"]], df[~df["in_corpus"]]
    src_hit = ind.apply(lambda r: r["source_doc_id"] in r["retrieved_doc_ids"], axis=1).mean()
    src_cited = ind.apply(lambda r: any(fuzz.token_set_ratio(t.lower(), r["source_title"].lower()) >= 85
                                        for t in r["cited_titles"]), axis=1).mean()
    n_cit = df["n_citations"].sum()
    return {"Abstention Accuracy": ood["refused"].mean(), "Hallucination Rate": 1 - ood["refused"].mean(),
            "False Refusal Rate": ind["refused"].mean(), "Source Hit@k": src_hit,
            "Citation Validity": df["n_valid_citations"].sum() / n_cit if n_cit else np.nan,
            "Source Cited": src_cited}

own_df = pd.DataFrame({f"top_k={k}": own_metrics(gen_runs[k]) for k in ABLATION_KS})
own_df.to_csv(RES_DIR / "generation_own_metrics.csv")
display(own_df.style.format(precision=3, na_rep="—"))
pd.concat(gen_runs.values()).drop(columns=["contexts"]).to_json(
    RES_DIR / "generation_runs.jsonl", orient="records", lines=True, force_ascii=False)

### 7.3. RAGAS: Faithfulness, Answer Relevancy, Context Precision, Context Recall

LLM-судья подключается через OpenAI-совместимый API. По умолчанию это Groq `llama-3.3-70b-versatile` (бесплатный
ключ на console.groq.com, сохранить в секреты Colab как `JUDGE_API_KEY`). Судья должен быть заметно сильнее
генератора: модель на 1,5 млрд параметров не справляется с разбором утверждений, которого требует RAGAS.

* **Faithfulness**: доля утверждений ответа, выводимых из `contexts` (обратная величина к галлюцинациям);
* **Answer Relevancy**: насколько ответ отвечает именно на заданный вопрос (через эмбеддинги вопросов, восстановленных из ответа);
* **Context Precision**: насколько высоко в выдаче стоят полезные фрагменты;
* **Context Recall**: покрывает ли контекст всё, что есть в эталонном ответе.

Если ключа нет, ячейка пропускается, а ablation строится по метрикам из 7.2.

In [ ]:
JUDGE_KEY = get_secret("JUDGE_API_KEY")
ragas_df = None
if JUDGE_KEY:
    try:
        !pip install -q "ragas==0.2.15" "langchain-openai>=0.3,<0.4" "langchain-huggingface>=0.1,<0.2" datasets
        from ragas import evaluate, EvaluationDataset, RunConfig
        from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from langchain_openai import ChatOpenAI
        from langchain_huggingface import HuggingFaceEmbeddings

        judge = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, base_url=JUDGE_BASE_URL, api_key=JUDGE_KEY,
                                               temperature=0, max_retries=6, timeout=120))
        judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMB_MODEL_MAIN))
        metrics = [Faithfulness(), ResponseRelevancy(), LLMContextPrecisionWithReference(), LLMContextRecall()]
        rows = []
        for k in ABLATION_KS:
            df = gen_runs[k]
            ds = EvaluationDataset.from_list([{"user_input": r["question"], "retrieved_contexts": r["contexts"],
                                               "response": r["answer"], "reference": r["ground_truth"]}
                                              for _, r in df.iterrows()])
            res = evaluate(ds, metrics=metrics, llm=judge, embeddings=judge_emb,
                           run_config=RunConfig(max_workers=2, timeout=180, max_retries=8), show_progress=True)
            res_df = res.to_pandas()
            res_df["k"], res_df["in_corpus"] = k, df["in_corpus"].values
            rows.append(res_df)
        ragas_df = pd.concat(rows, ignore_index=True)
        ragas_df.to_csv(RES_DIR / "ragas_scores.csv", index=False)
        metric_cols = [c for c in ragas_df.columns if c in
                       ("faithfulness", "answer_relevancy", "llm_context_precision_with_reference", "context_recall")]
        ragas_summary = ragas_df[ragas_df["in_corpus"]].groupby("k")[metric_cols].mean().T
        ragas_summary.columns = [f"top_k={c}" for c in ragas_summary.columns]
        ragas_summary.to_csv(RES_DIR / "ragas_summary.csv")
        print("RAGAS (вопросы с ответом в корпусе):")
        display(ragas_summary.style.format(precision=3))
    except Exception as e:
        print(f"RAGAS не выполнен: {type(e).__name__}: {e}")
else:
    print("JUDGE_API_KEY не задан — RAGAS пропущен, ablation строится по метрикам 7.2.")

### 7.4. Ablation Study: влияние объёма контекста

Ожидаемый компромисс: с ростом k растёт **полнота контекста** (Context Recall / Source Hit), но в промпт попадает больше
нерелевантных фрагментов. Небольшая модель начинает «смешивать» статьи, и **верность контексту** (Faithfulness /
Citation Validity) может падать. Это феномен информационного зашумления (*lost in the middle*).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
if ragas_df is not None and {"faithfulness", "context_recall"} <= set(ragas_df.columns):
    agg = ragas_df[ragas_df["in_corpus"]].groupby("k")[["faithfulness", "context_recall"]].mean()
    ax.plot(agg.index, agg["faithfulness"], marker="o", label="Faithfulness (RAGAS)", color="#4C72B0")
    ax.plot(agg.index, agg["context_recall"], marker="s", label="Context Recall (RAGAS)", color="#DD8452")
    ablation_source = "RAGAS"
else:
    ax.plot(ABLATION_KS, own_df.loc["Citation Validity"].values, marker="o", label="Citation Validity (прокси Faithfulness)", color="#4C72B0")
    ax.plot(ABLATION_KS, own_df.loc["Source Hit@k"].values, marker="s", label="Source Hit@k (прокси Context Recall)", color="#DD8452")
    ablation_source = "прокси-метрики"
ax.plot(ABLATION_KS, own_df.loc["Source Cited"].values, marker="^", ls="--", label="Source Cited", color="#55A868")
ax.set_xticks(ABLATION_KS); ax.set_xlabel("top_k"); ax.set_ylim(0, 1.05); ax.grid(alpha=0.3)
ax.set_title(f"Ablation по объёму контекста ({ablation_source})"); ax.legend()
plt.tight_layout(); savefig("07_ablation_topk"); plt.show()

## 8. Автоматический отчёт и выводы

Ячейка ниже собирает `REPORT.md` из **фактических** чисел этого прогона: таблицы, графики, сравнение сценариев
и выводы, сформулированные по результатам (какая стратегия лучше, насколько помогает очистка, как ведёт себя система
вне корпуса). Если ноутбук запущен из папки репозитория, в `README.md` обновляется блок с ключевыми результатами.

In [ ]:
def md(df, nd=3, index=True):
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_float_dtype(out[c]):
            out[c] = out[c].map(lambda v: "—" if pd.isna(v) else f"{v:.{nd}f}")
    return out.to_markdown(index=index)

def short(s, n=350):
    return textwrap.shorten(str(s).replace("\n", " ").replace("|", "/"), n)

g = grid_df.set_index(["модель", "стратегия"])
mA, mB = g.loc[(main_short, "A")], g.loc[(main_short, "B")]
mNT = g.loc[(main_short, "A-noTitle")]
other = [m.split("/")[-1] for m in EMB_MODELS_COMPARE if m != EMB_MODEL_MAIN]
best_model = grid_df.groupby("модель")["MRR@10"].mean().idxmax()
t_clean, t_noisy, t_fixed = typo_df.iloc[0], typo_df.iloc[1], typo_df.iloc[2]
k_best_src = own_df.loc["Source Hit@k"].astype(float).idxmax()

conclusions = [
    f"**Стратегии чанкинга.** На {len(eval_docs)} запросах-названиях лучшей по MRR@10 оказалась стратегия "
    f"**{BEST_STRATEGY}** (A: MRR@10 = {mA['MRR@10']:.3f}, Recall@5 = {mA['Recall@5']:.3f}; "
    f"B: MRR@10 = {mB['MRR@10']:.3f}, Recall@5 = {mB['Recall@5']:.3f}). "
    + ("Скользящее окно даёт чанки одинаковой длины с фиксированным перекрытием, поэтому каждый фрагмент получает "
       "сопоставимый «вес» в эмбеддинге." if BEST_STRATEGY == "A" else
       "Recursive splitter режет по границам предложений, поэтому фрагменты семантически цельные и эмбеддинг "
       "меньше «размывается» обрывками мыслей."),
    f"**Утечка через заголовок.** Без префикса `Title:` в хвостовых чанках (A-noTitle) Recall@5 меняется с "
    f"{mA['Recall@5']:.3f} до {mNT['Recall@5']:.3f}, а HitRate@1 — с {mA['HitRate@1']:.3f} до {mNT['HitRate@1']:.3f}. "
    "Префикс помогает находить *все* фрагменты статьи, а не только первый, то есть он полезен для полноты "
    "контекста; при этом совпадение названия частично упрощает задачу, и это нужно учитывать при интерпретации абсолютных значений.",
    f"**Модели эмбеддингов.** По среднему MRR@10 по стратегиям лучше модель **{best_model}** "
    f"(сравнивались {main_short} и {', '.join(other)}).",
    f"**Очистка запросов.** Recall@5: чистые названия {t_clean['Recall@5']:.3f} → с опечатками {t_noisy['Recall@5']:.3f} "
    f"→ после очистки {t_fixed['Recall@5']:.3f}; MRR@10: {t_clean['MRR@10']:.3f} → {t_noisy['MRR@10']:.3f} → "
    f"{t_fixed['MRR@10']:.3f}. Очистка вернула {t_fixed['MRR@10'] - t_noisy['MRR@10']:+.3f} MRR@10 из "
    f"{t_clean['MRR@10'] - t_noisy['MRR@10']:+.3f}, потерянных из-за опечаток. Точно восстановлено {word_restore_rate:.0%} искажённых слов, при этом все "
    "доменные термины из контрольного списка сохранены.",
    f"**Защита от галлюцинаций.** Abstention Accuracy на вопросах вне корпуса = "
    f"{own_df.loc['Abstention Accuracy'].mean():.0%} (среднее по top_k), Hallucination Rate = "
    f"{own_df.loc['Hallucination Rate'].mean():.0%}, False Refusal Rate на вопросах по теме = "
    f"{own_df.loc['False Refusal Rate'].mean():.0%}. Порог τ = {tau:.3f} отсекает посторонние вопросы ещё до LLM.",
    f"**Объём контекста.** Статья-источник попадает в контекст в {own_df.loc['Source Hit@k'].iloc[0]:.0%} случаев "
    f"при top_k=1 и в {own_df.loc['Source Hit@k'].iloc[-1]:.0%} при top_k=5; лучший Source Hit при {k_best_src}. "
    "Рост k повышает полноту контекста, но удлиняет промпт; влияние на верность ответа показано на графике ablation.",
]

scen_md = "| Сценарий | Вопрос | Чистая LLM | RAG | Автопроверка |\n|---|---|---|---|---|\n" + "\n".join(
    f"| {r['Сценарий']} | {short(r['Вопрос'], 120)} | {short(r['Ответ чистой LLM'])} | {short(r['Ответ RAG (LLM + FAISS)'])} | {short(r['Автопроверка'], 200)} |"
    for _, r in comparison_df.iterrows())

ragas_md = (md(ragas_summary) if ragas_df is not None and "ragas_summary" in globals()
            else "_RAGAS не запускался (нет `JUDGE_API_KEY`); см. метрики без судьи выше._")

report = f"""# Отчёт: RAG-поиск по научным статьям

**Тема корпуса:** {TOPIC_NAME}
**Генератор:** `{llm.name}` · **эмбеддинги:** `{EMB_MODEL_MAIN}` · **индекс:** FAISS `IndexFlatIP` (косинус)
**Дата прогона:** {time.strftime('%Y-%m-%d')} · seed = {SEED}

## 1. Корпус
{md(pd.Series(corpus_stats, name='значение').to_frame())}

![EDA](results/figures/01_corpus_eda.png)

## 2. Чанкинг
{md(chunk_stats_df.set_index('стратегия'), 2)}

![Длины чанков](results/figures/02_chunk_lengths.png)

## 3. Оценка поиска (N = {len(eval_docs)}, запрос = название статьи)
### Сетка экспериментов
{md(grid_df, index=False)}

### Стратегия A · {main_short}: метрика × k
{md(summaries[(main_short, 'A')])}

### Стратегия B · {main_short}: метрика × k
{md(summaries[(main_short, 'B')])}

![IR](results/figures/06_ir_strategies.png)

### Устойчивость к опечаткам (40 искажённых названий, индекс {BEST_STRATEGY})
{md(typo_df, index=False)}

Точно восстановлено слов: {restored}/{total} ({word_restore_rate:.0%}).

![Опечатки](results/figures/06_typo_robustness.png)

## 4. Пять тестовых сценариев: чистая LLM vs RAG
Порог уверенности τ = {tau:.3f} (точность разделения на калибровке {calib_acc:.0%}).

{scen_md}

Полные ответы: `results/scenarios_comparison.csv`.

## 5. Оценка генерации ({len(eval_set)} вопросов)
### Метрики без судьи
{md(own_df)}

### RAGAS (вопросы с ответом в корпусе)
{ragas_md}

![Ablation](results/figures/07_ablation_topk.png)

## 6. Выводы
""" + "\n".join(f"{i}. {c}" for i, c in enumerate(conclusions, 1)) + """

## 7. Ограничения и что можно улучшить
* Корпус состоит из аннотаций, а не полных текстов: на детальные вопросы (гиперпараметры, таблицы результатов) система честно отказывает.
* Офлайн-оценка с запросом-названием проще реальных пользовательских вопросов; следующий шаг — размеченный набор вопросов от людей.
* Улучшения: гибридный поиск (BM25 + dense) с Reciprocal Rank Fusion, cross-encoder реранкер (`bge-reranker`), HyDE-переформулировка запроса, генератор 7B+.
"""
Path("REPORT.md").write_text(report, encoding="utf-8")
print("REPORT.md сохранён\n")
for c in conclusions:
    print("•", c, "\n")

# ключевые результаты для README (обновляется, если ноутбук запущен из корня репозитория)
key_md = (f"| Метрика | Значение |\n|---|---|\n"
          f"| Статей в корпусе | {len(corpus_df)} |\n"
          f"| Лучшая стратегия чанкинга | {BEST_STRATEGY} |\n"
          f"| Recall@5 / MRR@10 (лучшая стратегия, {main_short}) | {g.loc[(main_short, BEST_STRATEGY), 'Recall@5']:.3f} / {g.loc[(main_short, BEST_STRATEGY), 'MRR@10']:.3f} |\n"
          f"| Recall@5 с опечатками: до → после очистки | {t_noisy['Recall@5']:.3f} → {t_fixed['Recall@5']:.3f} |\n"
          f"| Abstention Accuracy (вне корпуса) | {own_df.loc['Abstention Accuracy'].mean():.0%} |\n"
          f"| Hallucination Rate (вне корпуса) | {own_df.loc['Hallucination Rate'].mean():.0%} |\n")
(RES_DIR / "key_results.md").write_text(key_md, encoding="utf-8")
readme = Path("README.md")
if readme.exists():
    txt = readme.read_text(encoding="utf-8")
    new = re.sub(r"<!-- RESULTS:START -->.*<!-- RESULTS:END -->",
                 f"<!-- RESULTS:START -->\n{key_md}\n<!-- RESULTS:END -->", txt, flags=re.S)
    readme.write_text(new, encoding="utf-8")
    print("README.md: блок результатов обновлён")
print(key_md)

### Выгрузка результатов

Собираем всё, что нужно положить в репозиторий, в один архив: корпус, таблицы, графики и отчёт.
Сам ноутбук **с выводами** скачайте через `Файл → Скачать → Скачать .ipynb`.

In [ ]:
with zipfile.ZipFile("rag_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in [CORPUS_PATH, Path("REPORT.md"), *RES_DIR.rglob("*")]:
        if p.is_file():
            zf.write(p)
print("Архив: rag_results.zip", f"({Path('rag_results.zip').stat().st_size / 1e6:.1f} МБ)")
try:
    from google.colab import files
    files.download("rag_results.zip")
except Exception:
    pass